# core

> Kernel processes, channels, routing, and the gateway server

In [ ]:
#| default_exp core

In [ ]:
#| export
import asyncio, json, logging, os, signal, subprocess, sys, time, uuid
from collections import deque
from contextlib import asynccontextmanager
from pathlib import Path
from tempfile import mkdtemp
import zmq, zmq.asyncio
from jupywire.connect import write_connection_file
from fastcore.basics import xdumps, revive_dates
from fastcore.script import call_parse
from typing import Annotated
from jupywire.session import Session, pack_frames, unpack_frames
from starlette.applications import Starlette
from starlette.responses import JSONResponse, Response
from starlette.routing import Route, WebSocketRoute
from starlette.websockets import WebSocketDisconnect

In [ ]:
import httpx, struct, time
from fastcore.test import test_eq
from websockets.sync.client import connect as ws_connect

In [ ]:
#| export
log = logging.getLogger('jupygate')

## Wire formats

On the zmq side of the gateway, kernels speak the [Jupyter messaging protocol](https://jupyter-client.readthedocs.io/en/latest/messaging.html): each message is a dict with `header`, `parent_header`, `metadata`, `content`, and optional binary `buffers`. On the websocket side, the standard "legacy" Jupyter websocket protocol sends that same dict as JSON, with one extra `channel` key saying which zmq channel (`shell`, `iopub`, `stdin`, `control`) it belongs to. Adopting it unchanged means any Jupyter-compatible web client can talk to jupygate.

This first section is the codec: message dict plus channel in, websocket frame out, and back. It is pure functions over dicts and bytes, so it needs no kernel, no sockets, and no event loop.

### Text frames


A message without buffers becomes a single JSON text frame. `xdumps` handles the types Jupyter headers carry that JSON doesn't (datetimes, UUIDs); `revive_dates` reverses the datetime part on the way back in, applied only to the headers so a content string that merely looks like a date is left alone. This matches what jupyter_client's own `Session` does over zmq.


In [ ]:
#| export
def dumps(msg:dict)->str:
    "Encode a Jupyter message dict (no buffers) as a websocket text frame."
    return xdumps(msg)

def loads(s:str|bytes)->dict:
    "Decode a websocket text frame back into a message dict, reviving header dates."
    msg = json.loads(s)
    for k in ('header','parent_header'):
        if isinstance(msg.get(k), dict): msg[k] = revive_dates(msg[k])
    return msg


A realistic example to carry through this page: an `execute_request` as a client would send it, built with jupywire's `Session` (the shared jupyter_client mirror) so the header fields are the real thing.

In [ ]:
session = Session(key=b"secret")
req = session.msg('execute_request', dict(code='6*7', silent=False))
req['channel'] = 'shell'
sorted(req)

['channel',
 'content',
 'header',
 'metadata',
 'msg_id',
 'msg_type',
 'parent_header']

In [ ]:
frame = dumps(req)
frame[:120]

'{"header": {"msg_id": "73c7bf0b-4f1609c84c83a53ccee35599_31463_0", "msg_type": "execute_request", "username": "jhoward",'

The round trip preserves everything, including the parsed datetime in the header.

In [ ]:
back = loads(frame)
test_eq(back['header'], req['header'])
test_eq(back['content'], req['content'])
back['header']['date']

datetime.datetime(2026, 8, 2, 5, 59, 36, 260712, tzinfo=datetime.timezone.utc)

### Binary frames


Messages carrying `buffers` (comm messages with array data, for example) can't ride in JSON. The legacy protocol switches to a binary websocket frame with a small header: a 4-byte big-endian count `nbufs` (the JSON part plus each buffer), then `nbufs` 4-byte offsets from the start of the frame, then the JSON bytes, then the raw buffers back to back. This is the same layout jupyter_server uses, so existing web clients already decode it. The framing itself is `fastcore.nbio`'s `pack_frames`/`unpack_frames`; these wrappers just choose what goes in the JSON part.

In [ ]:
#| export
def dumps_binary(msg:dict)->bytes:
    "Encode a message with `buffers` as a legacy-protocol binary frame."
    msg = dict(msg)
    buffers = msg.pop('buffers')
    return pack_frames(dumps(msg).encode('utf-8'), buffers)

def loads_binary(bmsg:bytes)->dict:
    "Decode a legacy-protocol binary frame back into a message dict with `buffers`."
    body, buffers = unpack_frames(bmsg)
    msg = loads(body)
    msg['buffers'] = buffers
    return msg


A comm message with two buffers shows the layout. The first 12 bytes are the count (3 parts: JSON + 2 buffers) and two of the three offsets.

In [ ]:
comm = session.msg('comm_msg', dict(comm_id='abc', data={}))
comm['channel'] = 'iopub'
comm['buffers'] = [b'\x00'*8, b'payload']
bframe = dumps_binary(comm)
struct.unpack('!4I', bframe[:16])

(3, 16, 419, 427)

In [ ]:
back = loads_binary(bframe)
test_eq(back['buffers'], [b'\x00'*8, b'payload'])
test_eq(back['content'], comm['content'])
back['channel']

'iopub'

### One entry point each way


The gateway shouldn't care which encoding applies: `to_frame` picks based on whether buffers are present, and `from_frame` picks based on the frame type it was handed (websocket libraries deliver text frames as `str` and binary frames as `bytes`).

In [ ]:
#| export
def to_frame(msg:dict)->str|bytes:
    "Encode `msg` for the websocket: JSON text frame, or binary frame if it carries buffers."
    return dumps_binary(msg) if msg.get('buffers') else dumps({k:v for k,v in msg.items() if k!='buffers'})

def from_frame(data:str|bytes)->dict:
    "Decode a websocket frame (text or binary) into a message dict; `buffers` is always present."
    msg = loads_binary(data) if isinstance(data, (bytes,bytearray)) else loads(data)
    msg.setdefault('buffers', [])
    return msg

In [ ]:
test_eq(from_frame(to_frame(req))['content'], req['content'])
test_eq(from_frame(to_frame(comm))['buffers'], comm['buffers'])
type(to_frame(req)), type(to_frame(comm))

(str, bytes)

An empty `buffers` list stays a text frame, so clients that always attach the key pay no binary overhead.

In [ ]:
plain = dict(req, buffers=[])
assert isinstance(to_frame(plain), str)
from_frame(to_frame(plain))['buffers']

[]

## Kernel processes

The gateway is the kernel launcher. A client asks for a kernel with an argv template, an environment, and a working directory; the gateway writes a connection file, spawns the process, and later terminates it. There is no kernelspec lookup: since gateway clients are trusted (or token-authed), they say exactly what to run. That's what solveit needs for its custom launch (its own argv, env, cwd, and a gosu/sudo user wrap), and it removes the env-whitelisting machinery kernel_gateway grew to guard name-based spawning.

One consequence to state plainly: this API is arbitrary code execution by design. So is a Jupyter kernel. Auth (in `app`) is how a deployment that isn't localhost-only protects it.

### Connection files


A connection file is the small JSON dict every Jupyter kernel reads at startup: five ports, an ip, a transport, and the HMAC signing key. jupywire's `write_connection_file` picks free random ports and generates the file; we keep the returned info dict, which is what the channel layer (next module) needs to connect.

In [ ]:
#| export
def make_connection(dir:str|None=None)->tuple[str,dict]:
    "Write a connection file with random free ports and a fresh key; returns `(path, info)`."
    dir = dir or mkdtemp(prefix='jupygate-')
    fname = str(Path(dir)/f"kernel-{uuid.uuid4().hex[:8]}.json")
    return write_connection_file(fname, ip='127.0.0.1', key=uuid.uuid4().hex.encode())

In [ ]:
cf, info = make_connection()
{k:v for k,v in info.items() if k!='key'}

{'shell_port': 50224,
 'iopub_port': 50225,
 'stdin_port': 50226,
 'control_port': 50227,
 'hb_port': 50228,
 'ip': '127.0.0.1',
 'transport': 'tcp',
 'signature_scheme': 'hmac-sha256',
 'kernel_name': ''}

### KernelProc


`KernelProc` wraps one spawned kernel. The argv template uses `{connection_file}` the same way kernelspecs do; `python -m ipymini` is the argv we'll use throughout these notebooks (any Jupyter kernel works the same way). Two lifecycle details matter:

- `JPY_PARENT_PID` is set to our pid: kernels that watch it (ipymini does) shut themselves down if the gateway dies, so no orphans survive a gateway crash.
- `username` wraps the argv in `sudo -u` (or a gosu-style helper via `JUPYGATE_SUDO`), which is how solveit drops kernels to an unprivileged user in docker.

In [ ]:
#| export
def _sudo(username): 
    helper = os.environ.get('JUPYGATE_SUDO')
    return [helper, username] if helper else ['/usr/bin/sudo','-n','-E','-u',username]

class KernelProc:
    "One kernel process: spawn from an argv template, watch, terminate."
    def __init__(self, argv:list[str], env:dict|None=None, appendenv:dict|None=None, cwd:str|None=None, username:str|None=None):
        self.cfile, self.info = make_connection()
        cmd = [a.format(connection_file=self.cfile) for a in argv]
        if username:
            os.chmod(self.cfile, 0o644)
            cmd = [*_sudo(username), *cmd]
        full_env = dict(os.environ if env is None else env) | (appendenv or {})  # `env` replaces the inherited environment; `appendenv` overlays the base
        full_env['JPY_PARENT_PID'] = str(os.getpid())
        self.proc = subprocess.Popen(cmd, env=full_env, cwd=cwd, start_new_session=True)

    @property
    def pid(self): return self.proc.pid
    def alive(self)->bool: return self.proc.poll() is None
    def interrupt(self): os.kill(self.pid, signal.SIGINT)

    def terminate(self, timeout:float=5.0):
        "SIGTERM then SIGKILL; reaps the process and removes the connection file."
        if self.alive():
            self.proc.terminate()
            try: self.proc.wait(timeout)
            except subprocess.TimeoutExpired:
                self.proc.kill()
                self.proc.wait(timeout)
        Path(self.cfile).unlink(missing_ok=True)

In [ ]:
#| export
IPYMINI_ARGV = [sys.executable, '-m', 'ipymini', '-f', '{connection_file}']

In [ ]:
k = KernelProc(IPYMINI_ARGV)
k.alive(), k.pid != os.getpid()

(True, True)

The kernel needs a moment to boot before it binds its ports; proving it is alive and answering is the next section's job (that's what the ready-wait is for). Here we only exercise the process contract: it runs, and `terminate` reaps it and cleans up the connection file.

In [ ]:
k.terminate()
test_eq(k.alive(), False)
test_eq(Path(k.cfile).exists(), False)
k.proc.returncode is not None

True

A kernel that dies on its own is observable through the same `alive()`: `poll` reaps the zombie, so the gateway's kernel model can report an exited kernel without extra machinery.

In [ ]:
bad = KernelProc([sys.executable, '-c', 'raise SystemExit(3)'])
bad.proc.wait(5)
test_eq(bad.alive(), False)
bad.terminate()  # idempotent on a dead process
bad.proc.returncode

3

## Channels

jupyter_server opens fresh zmq streams for every websocket connection, which is why it needs its per-connect "nudge" dance and buffering handoffs. jupygate instead opens one channel set per *kernel*, at launch, and keeps it for the kernel's lifetime: a DEALER each for `shell`, `control`, and `stdin`, and a SUB for `iopub`. Websocket clients come and go without touching zmq at all.

Everything here is `zmq.asyncio` with awaited sends. That is a deliberate departure from jupyter_client's channel machinery, whose sync sends through a shadow of the asyncio socket consume the file-descriptor edge async receives park on (the bug conkernelclient exists to patch around). Awaited sends avoid it by construction.

`KernelChannels` connects to the five endpoints in a connection-info dict (we skip heartbeat: process liveness comes from `KernelProc.alive`, and responsiveness from the ready-wait below). The `Session`, built with the kernel's key, signs outbound messages and verifies inbound ones, so websocket clients never hold the key. The iopub SUB sets `RCVHWM=0`: with the default high-water mark of 1000, libzmq silently discards messages at the XPUB once the pipe fills during an output flood, statuses included - the kernel's never-drop-`status` policy lives above its socket, not below it - and a discarded `idle` leaves every client believing the kernel is busy forever. The gateway sheds in exactly one visible place, the per-client queue, so the zmq hop must be lossless; memory here is bounded in practice by the kernel's own iopub queue policy.

One non-obvious contract: the DEALER sockets must share a single zmq identity. Kernels address `input_request` on their stdin ROUTER to the identity of the *shell* request that called `input()`, assuming shell and stdin arrive from the same peer - jupyter_client guarantees that by giving both sockets its session id as identity, and we do the same.

In [ ]:
#| export
CHANNELS = ('shell','control','stdin','iopub')

class KernelChannels:
    "One persistent set of zmq.asyncio sockets connected to a kernel."
    def __init__(self, info:dict):
        self.session = Session(key=info['key'].encode() if isinstance(info['key'],str) else info['key'])
        self.ctx = zmq.asyncio.Context()
        addr = lambda port: f"{info['transport']}://{info['ip']}:{port}"
        self.socks = {}
        for name in CHANNELS:
            kind = zmq.SUB if name=='iopub' else zmq.DEALER
            s = self.ctx.socket(kind)
            s.linger = 0
            if kind==zmq.DEALER: s.setsockopt(zmq.IDENTITY, self.session.bsession)
            else:
                s.subscribe(b'')
                s.rcvhwm = 0  # before connect. Lossless hop: libzmq's HWM drops silently (statuses included); shedding belongs to ClientQueue alone
            s.connect(addr(info[f'{name}_port']))
            self.socks[name] = s

    async def send(self, channel:str, msg:dict):
        "Sign, serialize, and send a message dict on `channel`, awaiting the socket send so errors and backpressure surface here."
        m = dict(header=msg['header'], parent_header=msg.get('parent_header') or {}, metadata=msg.get('metadata') or {}, content=msg.get('content') or {})
        await self.socks[channel].send_multipart(self.session.serialize(m) + [memoryview(b) for b in msg.get('buffers') or []])

    async def recv(self, channel:str)->dict:
        "Receive and verify one message from `channel` (iopub topic frames are handled)."
        parts = await self.socks[channel].recv_multipart()
        idents, msg_list = self.session.feed_identities(parts)
        return self.session.deserialize(msg_list)

    def close(self):
        for s in self.socks.values(): s.close()
        self.ctx.term()

Wait: `Session.send` is a sync call on an asyncio socket - isn't that the exact bug described above? No: `Session.send` on a `zmq.asyncio` socket returns the send *future* after handing the frames to zmq, and crucially we never block a recv on the same socket edge, because DEALER replies and SUB traffic are read by dedicated recv calls on their own sockets. The conkernelclient bug needs a *sync shadow* socket doing the send; there is none here. (The regression test in conkernelclient's repo documents the failing combination.)

Time to talk to a real kernel. This kernel and channel set carry through the rest of the page.

In [ ]:
k = KernelProc(IPYMINI_ARGV)
ch = KernelChannels(k.info)
k.alive()

True

### The ready-wait


A SUB socket that connects while the kernel boots can miss early messages (zmq's slow-joiner problem). ipymini and modern ipykernel solve it with JEP 65: iopub is an XPUB socket, and each new subscription is greeted with an `iopub_welcome` message, after which nothing can be missed. But the welcome arrives at subscribe time, before any request/reply is possible, so support can't be negotiated - it can only be observed. And a welcome alone doesn't prove the kernel will *execute* anything: iopub comes up before the shell router, so "booting" and "wedged" look identical until a request round-trips.

So the ready-wait is one unified loop covering both worlds: send `kernel_info_request`, and watch iopub. Ready means the reply arrived *and* iopub is proven live - by a welcome or by any other iopub traffic. When the first iopub message is a welcome, one more `kernel_info` acts as an end marker: once its reply and its idle status arrive, everything the probing caused has been consumed, so readiness hands over provably clean channels. Any other first message means a pre-JEP-65 kernel, and the classic dance applies: take a reply, then drain iopub until it goes quiet.

In [ ]:
#| export
async def wait_ready(ch:KernelChannels, timeout:float=30.0, probe_every:float=0.5)->dict:
    "Block until the kernel answers `kernel_info` and iopub is proven live; returns the kernel_info reply content, leaving both channels clean."
    loop = asyncio.get_running_loop()
    end = loop.time() + timeout
    def left(cap=1.0):
        t = end - loop.time()
        if t <= 0: raise TimeoutError(f"kernel not ready after {timeout}s")
        return min(t, cap)
    async def recv_or_none(channel, cap=1.0):
        if await ch.socks[channel].poll(int(left(cap)*1000)): return await ch.recv(channel)
        return None
    probes = set()
    async def probe():
        m = ch.session.msg('kernel_info_request')
        probes.add(m['header']['msg_id'])
        await ch.send('shell', m)
        return m['header']['msg_id']
    msg = None
    while msg is None:
        await probe()
        msg = await recv_or_none('iopub', probe_every)
    if msg['header']['msg_type'] == 'iopub_welcome':
        mid = await probe()  # end marker: replies are FIFO, so once its reply and idle arrive, every earlier probe's traffic is consumed
        while (reply := await recv_or_none('shell')) is None or reply['parent_header'].get('msg_id') != mid: pass
        while not (msg['header']['msg_type']=='status' and msg['parent_header'].get('msg_id')==mid and msg['content']['execution_state']=='idle'):
            while (msg := await recv_or_none('iopub')) is None: pass
    else:
        while (reply := await recv_or_none('shell')) is None or reply['header']['msg_type'] != 'kernel_info_reply': pass
        outstanding = len(probes) - 1  # every probe gets a reply; one was consumed above
        while outstanding and await ch.socks['shell'].poll(1000):
            m = await ch.recv('shell')
            if m['parent_header'].get('msg_id') in probes: outstanding -= 1
        while await ch.socks['iopub'].poll(200): await ch.recv('iopub')  # then drain iopub until 0.2s of silence
    return reply['content']


In [ ]:
info = await wait_ready(ch)
assert not await ch.socks['iopub'].poll(300)  # ready leaves the channels clean: no probe traffic dribbles in afterwards
info['implementation'], info['status']


('ipymini', 'ok')

The fallback branch deserves its own pin. The same kernel started with `IPYMINI_IOPUB_XPUB=0` behaves like a pre-JEP-65 kernel: no welcome, so readiness comes from probe traffic and the silence drain, and the channels still come back clean:

In [ ]:
k3 = KernelProc(IPYMINI_ARGV, env=dict(os.environ, IPYMINI_IOPUB_XPUB='0'))
ch3 = KernelChannels(k3.info)
info = await wait_ready(ch3)
assert not await ch3.socks['iopub'].poll(300)
ch3.close()
k3.terminate()
info['status']

'ok'

With ipymini the proof of life is the welcome itself, and because our subscription predates everything the kernel ever published, it is still queued: the first iopub message on a fresh connection is the welcome.

In [ ]:
k2 = KernelProc(IPYMINI_ARGV)
ch2 = KernelChannels(k2.info)
first = await asyncio.wait_for(ch2.recv('iopub'), 30)
test_eq(first['header']['msg_type'], 'iopub_welcome')
first['content']

{'subscription': ''}

In [ ]:
#| hide
ch2.close()
k2.terminate()

### A cell, end to end


The channel set is enough to run code: send an `execute_request` on shell, read the busy/execute_input/stream/idle sequence on iopub, and the reply on shell. This is the traffic the mux (next module) will be fanning out to websocket clients.

In [ ]:
req = ch.session.msg('execute_request', dict(code='print(6*7)', silent=False, store_history=True,
    user_expressions={}, allow_stdin=False, stop_on_error=True))
await ch.send('shell', req)
seq = []
while True:
    m = await asyncio.wait_for(ch.recv('iopub'), 10)
    if m['parent_header'].get('msg_id') != req['header']['msg_id']: continue
    seq.append(m['header']['msg_type'])
    if m['header']['msg_type']=='status' and m['content']['execution_state']=='idle': break
seq

['status', 'execute_input', 'stream', 'status']

In [ ]:
reply = await asyncio.wait_for(ch.recv('shell'), 10)
test_eq(reply['header']['msg_type'], 'execute_reply')
test_eq(reply['content']['status'], 'ok')
reply['content']['execution_count']

1

Interrupts ride the control channel the same way (`interrupt_request` is what ipymini and ipykernel accept in-band; the HTTP layer also offers a SIGINT path for kernels configured for signal interrupts).

In [ ]:
imsg = ch.session.msg('interrupt_request', {})
await ch.send('control', imsg)
ireply = await asyncio.wait_for(ch.recv('control'), 10)
test_eq(ireply['header']['msg_type'], 'interrupt_reply')
ireply['content']

{'status': 'ok'}

## The mux

One kernel, many clients. The mux owns the kernel's channel set and a table of connected clients, and enforces four rules:

1. **iopub is broadcast**: every client sees all of it (that is the Jupyter model; outputs are public).
2. **shell/control replies go to their sender**: each client signs its requests with its own session id, replies echo it in `parent_header.session`, so a session-to-client map routes them with no per-message bookkeeping.
3. **stdin prompts go to the requester** the same way, and replies without a parent get one stamped on (below).
4. **a slow client hurts only itself**: each client has a bounded outbound queue; on overflow, non-`status` messages are dropped with a warning, `status` never, so every client's busy/idle picture stays truthful. This mirrors ipymini's own IOPub bound, and replaces libzmq's silent per-subscriber HWM drops with policy we can see and test.

A "client" here is anything with a `send(frame)` method and a `session_id`. The websocket adapter arrives in `app`; in this notebook clients are plain lists, which keeps every demonstration synchronous and inspectable.

### Client queues


`ClientQueue` wraps one client connection: a bounded deque of already-encoded frames drained by a writer task. The writer awaits the client's `send`, so a stalled websocket stalls only its own task while the deque absorbs (and, past the bound, sheds) the flood. Two details are load-bearing for reconnects: the writer sends the head frame and only pops it afterwards, so a send that dies mid-flight leaves the frame queued for a reattached connection instead of vanishing with the pop; and encoding happens at `put` time, so a frame handed to every client's queue is encoded exactly once by the iopub pump. The writer also yields to the event loop after every frame: a buffered transport can complete `send` after `send` without ever suspending, and an async loop that never suspends can neither be cancelled nor let the handler coroutine run to notice the disconnect - it would drain the whole ring into a dead socket. The one loss no queue discipline can close: a frame handed successfully to a socket whose network died underneath it is gone, and only acks and sequence numbers (which this protocol does not have) could recover it.


In [ ]:
#| export
class ClientQueue:
    "Outbound queue of encoded frames for one client. The bound governs iopub only, and never drops `status` while attached."
    def __init__(self, session_id:str, send, qmax:int=1000, buffer:bool=True):
        self.session_id, self.qmax, self.buffer = session_id, qmax, buffer
        self.q, self.dropped, self._drop_mark, self._wake = deque(), 0, 0, asyncio.Event()
        self.task = self.detached_at = None
        self.attach(send)

    @property
    def detached(self): return self.detached_at is not None

    def attach(self, send):
        "(Re)bind the queue to a live websocket: install `send` and start a fresh writer."
        if self.task: self.task.cancel()
        self._send, self.detached_at = send, None
        self.task = asyncio.create_task(self._writer())

    def detach(self):
        "Park the queue as a ring: stop the writer, remember when and how much was already dropped."
        if self.task: self.task.cancel()
        self.task, self.detached_at, self._drop_mark = None, time.monotonic(), self.dropped

    def put(self, msg:dict): self.put_frame(msg['header']['msg_type'], to_frame(msg), iopub=msg.get('channel')=='iopub')

    def put_frame(self, msg_type:str, frame:str|bytes, iopub:bool=True):
        if len(self.q) >= self.qmax and (self.detached or (iopub and msg_type!='status')):
            self.dropped += 1
            if self.dropped==1 or self.dropped%1000==0: log.warning("client %s queue full; dropped=%d", self.session_id, self.dropped)
            if not self.detached: return  # attached: the newcomer is dropped
            self.q.popleft()              # detached ring: the oldest is evicted, the newcomer kept
        self.q.append((msg_type, frame))
        self._wake.set()

    def force(self, msg:dict):
        "Queue `msg` past the bound (reattach synthesis: exactly two messages, never more)."
        self.q.append((msg['header']['msg_type'], to_frame(msg)))
        self._wake.set()

    async def _writer(self):
        while True:
            if not self.q:
                self._wake.clear()
                await self._wake.wait()
            try: await self._send(self.q[0][1])
            except Exception: return  # connection gone; the frame stays queued for a reattach
            self.q.popleft()
            await asyncio.sleep(0)  # a buffered transport can complete sends without suspending; yield so cancel and disconnect can land

    def close(self):
        if self.task: self.task.cancel()

`KernelMux` starts one pump task per channel, plus a reaper when buffering is on. The iopub pump encodes each message once and fans the frame out to every queue (it also tracks the latest `execution_state`, which reattach synthesis reports); shell, control, and stdin pumps look up the parent session. Messages for unknown sessions (a client whose queue is gone, or the gateway's own ready-probes) are dropped - over zmq they would simply have gone to a closed socket.


In [ ]:
#| export
class KernelMux:
    "Fan kernel traffic out to clients; route client frames back to kernel channels."
    def __init__(self, ch:KernelChannels, qmax:int=1000, buffer_secs:float=3600.0):
        self.ch, self.qmax, self.buffer_secs = ch, qmax, buffer_secs
        self.clients, self.pending_stdin, self.exec_state = {}, {}, 'starting'
        self.tasks = [asyncio.create_task(self._pump(c)) for c in CHANNELS]
        if buffer_secs: self.tasks.append(asyncio.create_task(self._reaper()))

    @property
    def n_attached(self): return sum(1 for cq in self.clients.values() if not cq.detached)

    def add(self, session_id:str, send, buffer:bool=True)->ClientQueue:
        "Register a client, or reattach a returning session to its surviving queue."
        cq = self.clients.get(session_id)
        if cq is None:
            self.clients[session_id] = cq = ClientQueue(session_id, send, self.qmax, buffer=buffer)
            return cq
        missed = cq.dropped - cq._drop_mark if cq.detached else 0
        cq.attach(send)
        if missed:
            warn = self.ch.session.msg('stream', dict(name='stderr', text=f'[jupygate] {missed} messages dropped while disconnected\n'))
            for m in (warn, self.ch.session.msg('status', dict(execution_state=self.exec_state))):
                m['channel'] = 'iopub'
                cq.force(m)
        return cq

    def drop(self, session_id:str):
        "A websocket went away: park the queue if the session is bufferable, else discard it."
        cq = self.clients.get(session_id)
        if cq is None: return
        if self.buffer_secs and cq.buffer: cq.detach()
        else:
            self.clients.pop(session_id)
            cq.close()

    async def _reaper(self):
        while True:
            await asyncio.sleep(min(self.buffer_secs, 60))
            cutoff = time.monotonic() - self.buffer_secs
            for sid, cq in list(self.clients.items()):
                if cq.detached and cq.detached_at < cutoff:
                    self.clients.pop(sid)
                    cq.close()

    async def _pump(self, channel:str):
        while True:
            msg = await self.ch.recv(channel)
            await asyncio.sleep(0)  # zmq recv on a buffered socket completes without suspending; yield so writers and handlers run mid-flood
            msg['channel'] = channel
            if channel=='iopub':
                if msg['header']['msg_type']=='status': self.exec_state = msg['content']['execution_state']
                mt, frame = msg['header']['msg_type'], to_frame(msg)
                for cq in self.clients.values(): cq.put_frame(mt, frame)
                continue
            sid = msg['parent_header'].get('session')
            if channel=='stdin' and msg['header']['msg_type']=='input_request': self.pending_stdin[sid] = msg['header']
            if (cq := self.clients.get(sid)): cq.put(msg)

    async def handle_frame(self, session_id:str, data:str|bytes):
        "One frame from a client websocket: decode, repair stdin parents, send to the kernel."
        msg = from_frame(data)
        channel = msg.pop('channel', 'shell')
        if channel=='stdin' and not msg.get('parent_header') and (hdr := self.pending_stdin.pop(session_id, None)):
            msg['parent_header'] = hdr
        await self.ch.send(channel, msg)

    def synthesize_status(self, state:str):
        "Broadcast a gateway-made status (`restarting`/`dead`): kernels cannot announce their own death."
        self.exec_state = state
        msg = self.ch.session.msg('status', dict(execution_state=state))
        msg['channel'] = 'iopub'
        mt, frame = 'status', to_frame(msg)
        for cq in self.clients.values(): cq.put_frame(mt, frame)

    def close(self):
        for t in self.tasks: t.cancel()
        for cq in self.clients.values(): cq.close()
        self.clients.clear()

### Two clients, one kernel


Two clients join the kernel we already have running: each is just a list collecting decoded frames, plus its own `Session` for signing, exactly as two websocket clients would look to the mux. First the helpers: a client factory, an execute-request builder, and a small poll-until.


In [ ]:
def mk_client():
    "A fake websocket client: a Session for signing, and a list collecting decoded frames."
    ses, got = Session(key=b'unused'), []
    async def send(frame): got.append(from_frame(frame))
    return ses, got, send

def exec_frame(ses, code):
    msg = ses.msg('execute_request', dict(code=code, silent=False, store_history=True,
        user_expressions={}, allow_stdin=True, stop_on_error=False))
    msg['channel'] = 'shell'
    return msg

async def until(pred, timeout=15):
    "Poll the fake clients until `pred()` is true."
    async with asyncio.timeout(timeout):
        while not pred(): await asyncio.sleep(0.02)


Client sessions sign with their own throwaway keys - the frames they produce are re-signed by the gateway's `Session` (which holds the kernel key) on the way in. That is the security boundary: the kernel key never leaves the gateway.

Both clients execute concurrently. Each gets its own reply; both see both cells' output on iopub.

In [ ]:
mux = KernelMux(ch)
ses_a, got_a, send_a = mk_client()
ses_b, got_b, send_b = mk_client()
mux.add(ses_a.session, send_a)
mux.add(ses_b.session, send_b)
await mux.handle_frame(ses_a.session, to_frame(exec_frame(ses_a, "a_val = 'from A'; print(a_val)")))
await mux.handle_frame(ses_b.session, to_frame(exec_frame(ses_b, "print('from B')")))
await until(lambda: any(m['channel']=='shell' for m in got_a) and any(m['channel']=='shell' for m in got_b))
[m['header']['msg_type'] for m in got_a if m['channel']=='shell'], [m['header']['msg_type'] for m in got_b if m['channel']=='shell']


(['execute_reply'], ['execute_reply'])

In [ ]:
replies_a = [m for m in got_a if m['channel']=='shell']
test_eq(replies_a[0]['parent_header']['session'], ses_a.session)  # A got A's reply, not B's
streams = {m['content']['text'].strip() for m in got_a if m['header']['msg_type']=='stream'}
assert {'from A','from B'} <= streams  # iopub is broadcast: A saw B's print too
streams

{'from A', 'from B'}

### Stdin and the parent-stamping repair


`input()` in a cell sends `input_request` to the requesting client. Most clients reply *without* a parent header (jupyter_client always has), which over raw zmq forces kernels into identity-matching heuristics - and through a shared-channel gateway even those break down. The mux repairs it at the source: it remembered the `input_request` header it delivered to this client, and stamps it onto a parentless `input_reply` in `handle_frame`. Our own websocket client sets the parent properly; the stamp is the safety net for everyone else's.

In [ ]:
await mux.handle_frame(ses_a.session, to_frame(exec_frame(ses_a, "answer = input('name? ')")))
await until(lambda: any(m['header']['msg_type']=='input_request' for m in got_a))
prompt = next(m for m in got_a if m['header']['msg_type']=='input_request')
prompt['content']

{'prompt': 'name? ', 'password': False}

In [ ]:
reply = ses_a.msg('input_reply', dict(value='Jeremy'))
reply['channel'] = 'stdin'   # note: no parent_header set - the worst-case client
await mux.handle_frame(ses_a.session, to_frame(reply))
await until(lambda: len([m for m in got_a if m['channel']=='shell']) >= 2)
await mux.handle_frame(ses_a.session, to_frame(exec_frame(ses_a, "print(answer)")))
await until(lambda: any('Jeremy' in m['content'].get('text','') for m in got_a if m['header']['msg_type']=='stream'))
next(m['content']['text'] for m in got_a if m['header']['msg_type']=='stream' and 'Jeremy' in m['content']['text'])

'Jeremy\n'

### Overflow: what a slow client misses, and what it never misses


A tiny queue bound and no writer draining it (we cancel the writer to simulate a stalled websocket) shows the shedding policy. The bound governs iopub output only: floods of `stream` messages drop, `status` always lands, and shell, control, and stdin messages are never shed at all, because they are paced by the client's own requests and a shed `execute_reply` would strand a reply-awaiting future forever. The client that eventually catches up may miss output, but it still knows the kernel went busy and idle again, and it always gets its replies:


In [ ]:
ses_c, got_c, send_c = mk_client()
cq = mux.add(ses_c.session, send_c)
cq.task.cancel()      # stall the writer: frames pile up in the deque
cq.qmax = 5
await mux.handle_frame(ses_c.session, to_frame(exec_frame(ses_c, "for i in range(200): print(i)")))
await until(lambda: cq.dropped > 0 and sum(t=='status' for t,_ in cq.q) >= 2)
types = [t for t,_ in cq.q]
cq.dropped, types.count('status'), len(cq.q) <= 5 + sum(t!='stream' for t in types), 'execute_reply' in types


client 44f7234b-69bc644e2d3d9d27f1b0578b queue full; dropped=1


(198, 2, True, True)

### Reconnects

A dropped websocket must not lose kernel output: the gateway sits across a real network, and the zmq side never drops (kernel channels are persistent), so the ws hop is the only place messages could vanish. When a connection goes away, `drop` therefore *detaches* the queue instead of discarding it: the writer stops, the deque stays behind as a bounded ring keyed by session id, and routing keeps filling it. A client that reconnects with the same `session_id` gets its queue back, and the ring drains before live traffic, in order, because it is one queue throughout.

Three policies bound this. Only sessions that supplied their own `session_id` are buffered: a gateway-generated id can never be presented again, so buffering it is pure waste (jupyasyncclient always supplies one). While detached the ring caps *everything*, statuses and replies included, evicting oldest so the newest window survives; the attached exemptions would otherwise grow without bound under a busy kernel. A reply evicted from a long-detached ring is not silent: the drop warning below says so, and a reply-awaiting client times out rather than hangs. And a ring detached for longer than `buffer_secs` (default an hour; 0 disables buffering) is reaped.

A client whose ring overflowed while it was away must not be silently missing output, so reattach appends two gateway-made messages after the ring: a stderr `stream` naming the count, and a `status` carrying the current execution state. The picture is explicitly, rather than accidentally, incomplete.

Client `a` disconnects (the ws handler calls `drop`), and a cell runs while it is away. Nothing reaches the dead socket, but the queue survives, detached, holding the traffic:

In [ ]:
mux.drop(ses_a.session)
n = len(got_a)
await mux.handle_frame(ses_b.session, to_frame(exec_frame(ses_b, "'while a is away'")))
await until(lambda: any(m['header']['msg_type']=='execute_result' for m in got_b))
qa = mux.clients[ses_a.session]
test_eq(len(got_a), n)
qa.detached, len(qa.q) > 0

(True, True)

Reattaching is `add` with the same session id: the surviving queue gets the new send function and drains, so `a`'s replacement connection sees the full output of the cell it missed, `execute_result` included:

In [ ]:
got_a2 = []
async def send_a2(frame): got_a2.append(from_frame(frame))
mux.add(ses_a.session, send_a2)
await until(lambda: any(m['header']['msg_type']=='execute_result' for m in got_a2))
next(m['content']['data']['text/plain'] for m in got_a2 if m['header']['msg_type']=='execute_result')

"'while a is away'"

Client `c` still has its tiny bound from the overflow demo. Detached, its ring keeps only the newest window, and reattach appends the two honesty messages: the stderr count of what was missed, then the current execution state:

In [ ]:
mux.drop(ses_c.session)
nb = len(got_b)
await mux.handle_frame(ses_c.session, to_frame(exec_frame(ses_c, "for i in range(200): print(i)")))
await until(lambda: any(m['header']['msg_type']=='status' and m['content']['execution_state']=='idle' for m in got_b[nb:]))
got_c2 = []
async def send_c2(frame): got_c2.append(from_frame(frame))
cq = mux.add(ses_c.session, send_c2)
await until(lambda: not cq.q)
warn = next(m for m in got_c2 if m['header']['msg_type']=='stream' and 'dropped while disconnected' in m['content']['text'])
test_eq(got_c2[-1]['header']['msg_type'], 'status')
len(got_c2), warn['content']['text'].strip()

(10, '[jupygate] 204 messages dropped while disconnected')

And the synthesized lifecycle statuses reach everyone, because a dead kernel cannot say goodbye itself:

In [ ]:
mux.synthesize_status('dead')
await until(lambda: any(m['content'].get('execution_state')=='dead' for m in got_a2 if m['header']['msg_type']=='status'))
sum(1 for got in (got_a2, got_b) for m in got if m['content'].get('execution_state')=='dead')

2

In [ ]:
#| hide
mux.close()
ch.close()
k.terminate()

## The gateway server

The last layer is thin by design: a starlette app whose routes are jupyter_server's kernels API (so existing clients work unchanged), backed by one `GatewayKernel` (process + channels + mux) per kernel id. The service returns JSON and frames only - no HTML, no files API (files are managed by running Python in a kernel), no kernelspecs (creation takes explicit argv/env/cwd).

- `GET /api/kernels` - list; `POST /api/kernels` - create; `GET|DELETE /api/kernels/{id}` - model / shutdown
- `POST /api/kernels/{id}/interrupt` and `/restart`
- `WS /api/kernels/{id}/channels?session_id=...` - the mux

Auth is optional: pass `auth_token` and every request must carry it (`Authorization: token ...` header, or `?token=` for websockets); localhost deployments can run open.

### GatewayKernel


`GatewayKernel` ties the three lower layers together, and its `start` is the only place the ready-wait runs: once per kernel, ever. `watch` polls the process and, if it dies unexpectedly, broadcasts the synthesized `dead` status. `restart` terminates and respawns; the new process gets fresh ports, so the channel set is rebuilt and clients simply see `restarting` then a fresh welcome-backed ready kernel.

In [ ]:
#| export
class GatewayKernel:
    "A kernel process plus its channel set and mux, as one lifecycle."
    def __init__(self, argv:list[str], env:dict|None=None, appendenv:dict|None=None, cwd:str|None=None,
        username:str|None=None, qmax:int=1000, buffer_secs:float=3600.0):
        self.spec = dict(argv=argv, env=env, appendenv=appendenv, cwd=cwd, username=username)
        self.qmax, self.buffer_secs = qmax, buffer_secs
        self.id = uuid.uuid4().hex
        self.proc = self.ch = self.mux = self._watcher = None
        self.state = 'starting'

    async def start(self, timeout:float=60.0):
        self.proc = KernelProc(**self.spec)
        self.ch = KernelChannels(self.proc.info)
        await wait_ready(self.ch, timeout)
        self.mux = KernelMux(self.ch, qmax=self.qmax, buffer_secs=self.buffer_secs)
        self.state = 'alive'
        self._watcher = asyncio.create_task(self._watch())
        return self

    async def _watch(self):
        while self.proc.alive(): await asyncio.sleep(0.5)
        if self.state=='alive':
            self.state = 'dead'
            self.mux.synthesize_status('dead')

    def model(self)->dict: return dict(id=self.id, name='', execution_state=self.state, connections=self.mux.n_attached if self.mux else 0)
    def interrupt(self): self.proc.interrupt()

    async def restart(self, timeout:float=60.0):
        self.state = 'restarting'
        self.mux.synthesize_status('restarting')
        old_mux = self.mux
        clients, old_mux.clients = old_mux.clients, {}
        await self.shutdown(keep_state=True)
        old_mux.close()
        await self.start(timeout)
        self.mux.clients = clients
        self.mux.synthesize_status('starting')

    async def shutdown(self, keep_state:bool=False):
        if self._watcher: self._watcher.cancel()
        if not keep_state:
            self.state = 'dead'
            if self.mux: self.mux.close()
        if self.ch: self.ch.close()
        if self.proc: await asyncio.to_thread(self.proc.terminate)

Restart keeps the *clients*: their queues are re-registered on the new mux, so an open websocket rides through a restart with no reconnect. What they see is deterministic and gateway-made: `restarting` immediately, then `starting` once the new kernel has passed its ready-wait. (The new kernel's own boot traffic, its welcome included, is consumed by the gateway's ready-wait before clients are re-attached, so the gateway speaks for the boot rather than racing to relay it.)


### The kernel registry

The lifecycle bookkeeping is its own object because the HTTP app is only one consumer of it. An embedder - an MCP server reusing these internals, say - drives the same registry directly and attaches to a kernel's mux exactly the way the fake clients above did, which gives it reply routing, stdin stamping, and the synthesized lifecycle statuses without any websocket in sight. The app below is routes over this object, plus auth. `create` owns the one non-trivial path: a kernel that fails to start is removed and torn down before the error propagates, so the registry never holds a half-alive entry.

In [ ]:
#| export
class Kernels:
    "Kernel registry: create, look up, and reap `GatewayKernel`s. The HTTP app and embedders both drive this."
    def __init__(self, argv:list[str]=IPYMINI_ARGV, qmax:int=1000, buffer_secs:float=3600.0):
        self.argv, self.qmax, self.buffer_secs = argv, qmax, buffer_secs
        self.kernels = {}

    async def create(self, argv:list[str]|None=None, env:dict|None=None, appendenv:dict|None=None,
        cwd:str|None=None, username:str|None=None, timeout:float=60.0)->GatewayKernel:
        gk = GatewayKernel(argv=argv or self.argv, env=env, appendenv=appendenv, cwd=cwd, username=username, qmax=self.qmax, buffer_secs=self.buffer_secs)
        self.kernels[gk.id] = gk
        try: await gk.start(timeout)
        except Exception:
            self.kernels.pop(gk.id, None)
            await gk.shutdown()
            raise
        return gk

    async def delete(self, kid:str):
        gk = self.kernels.pop(kid)
        await gk.shutdown()

    async def shutdown(self):
        "Reap every kernel; nothing survives the registry."
        await asyncio.gather(*[k.shutdown() for k in self.kernels.values()], return_exceptions=True)
        self.kernels.clear()

    def get(self, kid:str)->GatewayKernel|None: return self.kernels.get(kid)
    def values(self): return self.kernels.values()
    def __getitem__(self, kid:str)->GatewayKernel: return self.kernels[kid]
    def __len__(self): return len(self.kernels)

In [ ]:
kernels = Kernels()
gk = await kernels.create()
ses, got, send = mk_client()
gk.mux.add(ses.session, send, buffer=False)   # an in-process client: no websocket, no reconnect story
await gk.mux.handle_frame(ses.session, to_frame(exec_frame(ses, "6*7")))
await until(lambda: any(m['channel']=='shell' for m in got))
test_eq(kernels[gk.id], gk)
await kernels.shutdown()
test_eq(len(kernels), 0)

In [ ]:
#| export
def _authed(request, token):
    if not token: return True
    hdr = request.headers.get('authorization', '')
    return hdr == f'token {token}' or request.query_params.get('token') == token

def create_app(argv:list[str]=IPYMINI_ARGV, auth_token:str|None=None, qmax:int=1000, buffer_secs:float=3600.0)->Starlette:
    "The gateway app. `argv` is the default kernel command; creation requests may override it."
    kernels = Kernels(argv, qmax=qmax, buffer_secs=buffer_secs)

    def _kernel(request):
        k = kernels.get(request.path_params['kid'])
        if k is None: raise KeyError
        return k

    async def list_kernels(request): return JSONResponse([k.model() for k in kernels.values()])

    async def create_kernel(request):
        body = await request.json() if await request.body() else {}
        try: gk = await kernels.create(argv=body.get('argv'), env=body.get('env'), appendenv=body.get('appendenv'), cwd=body.get('cwd'), username=body.get('username'))
        except Exception as e: return JSONResponse(dict(message=f'kernel failed to start: {e}'), status_code=500)
        return JSONResponse(gk.model(), status_code=201)

    async def get_kernel(request): return JSONResponse(_kernel(request).model())

    async def delete_kernel(request):
        await kernels.delete(request.path_params['kid'])
        return Response(status_code=204)

    async def interrupt(request):
        _kernel(request).interrupt()
        return Response(status_code=204)

    async def restart(request):
        gk = _kernel(request)
        await gk.restart()
        return JSONResponse(gk.model())

    async def channels(ws):
        if not _authed(ws, auth_token): return await ws.close(code=4403)
        gk = kernels.get(ws.path_params['kid'])
        if gk is None or gk.mux is None: return await ws.close(code=4404)
        sid = ws.query_params.get('session_id')
        sid, buffer = sid or uuid.uuid4().hex, sid is not None
        await ws.accept()
        async def send(frame):
            if isinstance(frame, bytes): await ws.send_bytes(frame)
            else: await ws.send_text(frame)
        gk.mux.add(sid, send, buffer=buffer)
        try:
            while True:
                event = await ws.receive()
                if event['type']=='websocket.disconnect': break
                try: await gk.mux.handle_frame(sid, event.get('bytes') or event['text'])
                except Exception as e: log.warning('dropped frame from %s: %s', sid, e)
        except WebSocketDisconnect: pass
        finally: gk.mux.drop(sid)

    def guard(fn):
        async def inner(request):
            if not _authed(request, auth_token): return JSONResponse(dict(message='forbidden'), status_code=403)
            try: return await fn(request)
            except KeyError: return JSONResponse(dict(message='no such kernel'), status_code=404)
        return inner

    @asynccontextmanager
    async def lifespan(app):
        yield
        await kernels.shutdown()

    r = lambda p,meth,f: Route('/api/kernels'+p, guard(f), methods=[meth])
    app = Starlette(lifespan=lifespan, routes=[r('','GET',list_kernels), r('','POST',create_kernel), r('/{kid}','GET',get_kernel),
        r('/{kid}','DELETE',delete_kernel), r('/{kid}/interrupt','POST',interrupt), r('/{kid}/restart','POST',restart),
        WebSocketRoute('/api/kernels/{kid}/channels', channels)])
    app.state.kernels = kernels
    return app

### A live gateway


Serving the app the way solveit's notebooks serve fasthtml apps: uvicorn in a daemon thread with its own event loop. The gateway serving this section keeps running to the end of the page.

In [ ]:
#| export
def serve(app, host:str='127.0.0.1', port:int=8787, log_level:str='warning', in_thread:bool=False):
    "Run the gateway under uvicorn; `in_thread=True` waits until it is listening and returns the server with `server.url` set (`port=0` picks a free port)."
    import threading, time, uvicorn
    server = uvicorn.Server(uvicorn.Config(app, host=host, port=port, log_level=log_level, ws_max_size=64*2**20))
    if not in_thread:
        server.run()
        return server
    server.thread = threading.Thread(target=server.run, daemon=True)
    server.thread.start()
    end = time.monotonic() + 10
    while not server.started:
        if time.monotonic() > end: raise TimeoutError('uvicorn did not start')
        time.sleep(0.01)
    server.url = f'http://{host}:{server.servers[0].sockets[0].getsockname()[1]}'
    return server

def _env_app():
    "App factory for the reloader: its fresh workers re-import this module, so config rides in the environment; the token is popped so spawned kernels never inherit it"
    return create_app(auth_token=os.environ.pop('JUPYGATE_TOKEN', None))

def _reload_file():
    "The restart lever: touching this file restarts the gateway, killing all kernels; created at startup so it is always touchable"
    from fastcore.xdg import xdg_state_home
    p = xdg_state_home()/'jupygate'/'reload'/'r.py'
    p.parent.mkdir(parents=True, exist_ok=True)
    p.touch()
    return p

@call_parse
def main(
    host:str='127.0.0.1', # Interface to bind
    port:int=8787, # Port to listen on
    token:str=None, # Auth token clients must present
    log_level:Annotated[str,'Uvicorn log level',{'choices':['critical','error','warning','info','debug','trace']}]='warning',
    reload:bool=False # also restart on package source changes (dev)
):
    "Websocket gateway for Jupyter kernels"
    import uvicorn
    from uvicorn.supervisors import ChangeReload
    if token: os.environ['JUPYGATE_TOKEN'] = token
    reload_dirs = [str(_reload_file().parent)]
    if reload: reload_dirs.append(str(Path(__file__).parent))
    config = uvicorn.Config('jupygate.core:_env_app', factory=True, reload=True, reload_dirs=reload_dirs,
        host=host, port=port, log_level=log_level, ws_max_size=64*2**20, timeout_graceful_shutdown=5)
    server = uvicorn.Server(config)
    try: ChangeReload(config, target=server.run, sockets=[config.bind_socket()]).run()
    except KeyboardInterrupt: pass

In [ ]:
server = serve(create_app(), port=0, in_thread=True)
base = server.url
server.started, base


(True, 'http://127.0.0.1:50454')

The REST surface, in the order a client would use it: nothing running, create one (this boots a real ipymini and waits for its welcome), see it listed.

In [ ]:
http = httpx.Client(base_url=base, timeout=90)
test_eq(http.get('/api/kernels').json(), [])
kid = http.post('/api/kernels').json()['id']
http.get(f'/api/kernels/{kid}').json()

{'id': '7ec54351760647ec865c4f896af349c8',
 'name': '',
 'execution_state': 'alive',
 'connections': 0}

Executing over the websocket needs only the wire codec from the first section - which is the point: this is the same frame any Jupyter web client sends. One property to design for: each channel is ordered, but ordering *across* channels is not guaranteed (the reply can arrive before the last iopub output), same as over raw zmq.

In [ ]:
ses = Session(key=b'unused')
ws = ws_connect(f"{base.replace('http','ws')}/api/kernels/{kid}/channels?session_id={ses.session}")
msg = ses.msg('execute_request', dict(code='21*2', silent=False, store_history=True,
    user_expressions={}, allow_stdin=False, stop_on_error=True))
msg['channel'] = 'shell'
ws.send(to_frame(msg))
result = reply = None
while not (result and reply):
    m = from_frame(ws.recv(timeout=15))
    if m['header']['msg_type']=='execute_result': result = m['content']['data']['text/plain']
    if m['channel']=='shell': reply = m['content']['status']
result, reply


('42', 'ok')

Interrupt has two paths, and both work here: in-band `interrupt_request` on the control channel rides the same websocket; the HTTP endpoint sends SIGINT for clients that don't hold a control channel (or kernels that only support signal interrupts). We use the HTTP one to break an infinite loop:

In [ ]:
msg = ses.msg('execute_request', dict(code='import time\nwhile True: time.sleep(0.1)', silent=False,
    store_history=True, user_expressions={}, allow_stdin=False, stop_on_error=True))
msg['channel'] = 'shell'
ws.send(to_frame(msg))
time.sleep(0.5)
test_eq(http.post(f'/api/kernels/{kid}/interrupt').status_code, 204)
while True:
    m = from_frame(ws.recv(timeout=15))
    if m['channel']=='shell': break
m['content']['status'], m['content'].get('ename')

('error', 'KeyboardInterrupt')

Restart rides the same open websocket: `restarting`, then `starting` when the fresh kernel is ready, with no reconnect.


In [ ]:
test_eq(http.post(f'/api/kernels/{kid}/restart').status_code, 200)
states = []
while 'starting' not in states:
    m = from_frame(ws.recv(timeout=30))
    if m['header']['msg_type']=='status': states.append(m['content']['execution_state'])
assert 'restarting' in states
states[-2:]


['restarting', 'starting']

In [ ]:
ws.close()
test_eq(http.delete(f'/api/kernels/{kid}').status_code, 204)
test_eq(http.get('/api/kernels').json(), [])

Environment control is a pair whose names document each other: `env` *replaces* the kernel's environment wholesale (the cross-user case wants a curated env, not the gateway's), while `appendenv` *overlays* the inherited one. The overlay matters most over the network: an HTTP client wanting "the gateway's normal environment plus this one var" cannot build that dict itself, since it can't read the remote gateway's environment — only the server can merge:

In [ ]:
kid = http.post('/api/kernels', json=dict(appendenv=dict(JUPYGATE_DEMO_MARK='overlaid'))).json()['id']
ses2 = Session(key=b'unused')
ws = ws_connect(f"{base.replace('http','ws')}/api/kernels/{kid}/channels?session_id={ses2.session}")
msg = ses2.msg('execute_request', dict(code='import os; (os.environ["JUPYGATE_DEMO_MARK"], "PATH" in os.environ)',
    silent=False, store_history=True, user_expressions={}, allow_stdin=False, stop_on_error=True))
msg['channel'] = 'shell'
ws.send(to_frame(msg))
result = None
while not result:
    m = from_frame(ws.recv(timeout=15))
    if m['header']['msg_type']=='execute_result': result = m['content']['data']['text/plain']
ws.close()
test_eq(http.delete(f'/api/kernels/{kid}').status_code, 204)
test_eq(result, "('overlaid', True)")

### Auth


With a token configured, HTTP without it is 403 and the websocket closes immediately; both header and query-param forms are accepted.

In [ ]:
server2 = serve(create_app(auth_token='sekret'), port=0, in_thread=True)
open_http = httpx.Client(base_url=server2.url, timeout=30)
test_eq(open_http.get('/api/kernels').status_code, 403)
test_eq(open_http.get('/api/kernels', headers={'Authorization':'token sekret'}).status_code, 200)
test_eq(open_http.get('/api/kernels', headers={'Authorization':'token wrong'}).status_code, 403)
test_eq(open_http.get('/api/kernels', params={'token':'sekret'}).status_code, 200)
open_http.get('/api/kernels', headers={'Authorization':'token sekret'}).json()

[]

In [ ]:
#| hide
server.should_exit = server2.should_exit = True

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()